## 1.2 Cleaning and converting the articles

The probe (1.1) settled how to recover each piece of an article: the true source, the true publish time, and a clean body. This notebook turns those findings into one pipeline, built stage by stage, and runs it over the open corpus to produce the processed dataset. The exploratory analysis behind each stage lives in 1.1; here we keep the final code and a short note per stage.

### 1. Overview and setup

One fetch per article, parsed once: the pipeline reads the source, time, and body from a single page, and only fetches again to follow a canonical link. Setup loads the corpus, fixes the readable source set from the probe, and defines the shared pieces every stage uses, a pooled retrying session and the JSON-LD reader.

In [14]:
import json
import re
import random
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import timezone
from urllib.parse import urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dateutil import parser as date_parser
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from stock_predictor.config import RAW_DATA_DIR, PROCESSED_DATA_DIR

articles = pd.read_parquet(RAW_DATA_DIR / "raw_articles.parquet")
open_sources = ["Yahoo", "Benzinga", "DowJones"]  # readable sources found in the probe (1.1)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}
MAX_WORKERS = 8
TIMEOUT = 12
MIN_BODY_CHARS = 500
MAX_SHIFT_HOURS = 6  # a scraped time this far from the API time is treated as a repost stamp
REQ_PER_SEC = 3.5   # global fetch rate: fast enough to be quick, gentle enough to sit under Yahoo's 429 limit


def make_session():
    """A pooled, retrying session shared across the worker threads."""
    s = requests.Session()
    s.headers.update(HEADERS)
    # Bounded backoff, and ignore the server's Retry-After: under load Yahoo
    # returns a long Retry-After that parks a worker for tens of seconds, which
    # is what stalled the corpus run. backoff_max caps each wait instead.
    retry = Retry(total=2, backoff_factor=0.5, backoff_max=8,
                  status_forcelist=[429, 500, 502, 503, 504],
                  respect_retry_after_header=False)
    adapter = HTTPAdapter(max_retries=retry, pool_connections=MAX_WORKERS,
                          pool_maxsize=MAX_WORKERS * 2)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s


class Pacer:
    """Cap the request rate across all worker threads to REQ_PER_SEC.

    The pool bursts every request at once, which empties Yahoo's rate-limit
    bucket in seconds and trips the 429 wall. The pacer hands out a start slot
    per request so the whole pool fetches at a steady rate the limiter tolerates.
    """
    def __init__(self, per_second):
        self._interval = 1.0 / per_second
        self._lock = threading.Lock()
        self._next = 0.0

    def wait(self):
        with self._lock:
            now = time.monotonic()
            start = max(now, self._next)
            self._next = start + self._interval
        time.sleep(start - now)


pacer = Pacer(REQ_PER_SEC)


def _ld_objects(soup):
    """Yield each JSON-LD object on the page, flattening list payloads."""
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(tag.string or "")
        except (json.JSONDecodeError, TypeError):
            continue
        for obj in (data if isinstance(data, list) else [data]):
            if isinstance(obj, dict):
                yield obj


print(f"Loaded {len(articles)} articles")

Loaded 3143 articles


## Building the pipeline by stage

Each stage is one small function reading from the parsed page. They are composed into a single per-article pass in section 5.

### 2. Stage 1: the source

Take the syndication provider named in the page's structured data, then the publisher, then fall back to the final host. The provider is what recovers the real outlet hidden under a Yahoo repost. Full analysis in 1.1, section 3.2.

In [15]:
def _true_source(soup, host):
    """Prefer the syndication provider, then the publisher, then the host."""
    provider = publisher = None
    for obj in _ld_objects(soup):
        prov = obj.get("provider")
        if provider is None and prov:
            provider = prov.get("name") if isinstance(prov, dict) else (prov if isinstance(prov, str) else None)
        pub = obj.get("publisher")
        if publisher is None and isinstance(pub, dict) and pub.get("name"):
            publisher = pub["name"]
    return provider or publisher or host

### 3. Stage 2: the time

Read `datePublished` from the structured data and parse it to UTC; a zoneless time is not trusted. On a repost the wrapper time can be a repost or refresh stamp, so the canonical link is followed to the original publisher for the true first publish (the only place a second fetch happens).

The recovered time is then treated as a candidate against the Finnhub API time, not a blind overwrite. A cross-host original is the outlet's own page, so it is trusted outright; a same-host time is trusted only when it sits within `MAX_SHIFT_HOURS` of the API time; otherwise the API time stands. This keeps the genuine corrections, the Eastern-offset shift and the cross-host wins, while rejecting the wild repost stamps, and it makes the API time a floor no article falls below. Full analysis in 1.1, sections 3.4 and 3.6.

In [16]:
def _published(soup):
    """Read the published time from JSON-LD, or fall back to a meta tag."""
    for obj in _ld_objects(soup):
        if obj.get("datePublished"):
            return obj["datePublished"]
    m = soup.find("meta", attrs={"property": "article:published_time"})
    return m["content"] if m and m.get("content") else None


def _to_utc(value):
    """Parse an ISO time to UTC. Naive (zoneless) times are not trusted."""
    if not value:
        return None
    try:
        dt = date_parser.parse(value)
    except (ValueError, TypeError, OverflowError):
        return None
    if dt.tzinfo is None:
        return None
    return dt.astimezone(timezone.utc)


def _resolve_time(candidate, api_utc, cross_host):
    """Pick the trustworthy timestamp, with the API time as the floor.

    A cross-host canonical is the original publisher's own page, so its time is
    taken outright. A same-host time is taken only when it sits within
    MAX_SHIFT_HOURS of the API time, which keeps the Eastern-offset correction
    and small drift while rejecting repost stamps. Otherwise the Finnhub API
    time stands, so no article is dropped for want of a page time.
    """
    if candidate is None:
        return api_utc, "api"
    if cross_host:
        return candidate, "corrected"
    if abs((candidate - api_utc).total_seconds()) <= MAX_SHIFT_HOURS * 3600:
        return candidate, "corrected"
    return api_utc, "api"

### 4. Stage 3: the body

Drop the script, style, and chrome tags, join the paragraph text, and collapse whitespace. An article is kept only when the cleaned body clears `MIN_BODY_CHARS`; short stubs and consent shells fall out here.

In [17]:
def _clean_body(soup):
    """Drop chrome tags, join the paragraph text, collapse whitespace."""
    for tag in soup(["script", "style", "nav", "aside", "footer", "header", "form", "figure"]):
        tag.decompose()
    text = " ".join(p.get_text(" ", strip=True) for p in soup.find_all("p"))
    return re.sub(r"\s+", " ", text).strip()

### 5. Composing the stages

`process_article` runs the three stages on one page and records a pass or fail for each. The time is resolved against the Finnhub API time (`_resolve_time`), so every reached article ends with a usable time and the body becomes the only gate. `run_pipeline` maps it across the corpus on a shared session with an 8-worker pool. An article is accepted when its body clears the bar; a missing source is allowed, since it falls back to the host.

In [18]:
def process_article(row, session):
    """Fetch one article once and derive source, UTC time, and body.

    The time is resolved against the Finnhub API time: a cross-host canonical is
    trusted, a same-host time only within MAX_SHIFT_HOURS, else the API time
    stands. A second fetch fires only to follow a cross-host canonical link.
    """
    out = {"article_id": row["article_id"], "true_source": None, "utc": None,
           "time_source": None, "raw_body": None, "processed_body": None,
           "status": None, "source_ok": False, "corrected": False,
           "text_ok": False, "n_fetch": 0}
    url = row["url"]
    try:
        pacer.wait()  # hold to the global rate so the pool does not trip Yahoo's 429 limit
        r = session.get(url, timeout=TIMEOUT)
        out["n_fetch"] = 1
        out["status"] = r.status_code
        if r.status_code != 200:
            return out
        host = urlparse(r.url).netloc
        soup = BeautifulSoup(r.text, "html.parser")

        out["true_source"] = _true_source(soup, host)
        out["source_ok"] = out["true_source"] is not None

        published = _published(soup)
        cross_host = False
        link = soup.find("link", rel="canonical")
        canonical = link["href"] if link and link.get("href") else None
        if canonical and urlparse(canonical).netloc not in ("", host):
            try:
                r2 = session.get(canonical, timeout=TIMEOUT)
                out["n_fetch"] = 2
                if r2.status_code == 200:
                    original = _published(BeautifulSoup(r2.text, "html.parser"))
                    if original:
                        published, cross_host = original, True
            except requests.RequestException:
                pass

        candidate = _to_utc(published)
        api_utc = row["timestamp_utc"].to_pydatetime()
        out["utc"], out["time_source"] = _resolve_time(candidate, api_utc, cross_host)
        out["corrected"] = out["time_source"] == "corrected"

        out["raw_body"] = r.text
        out["processed_body"] = _clean_body(soup)
        out["text_ok"] = len(out["processed_body"]) >= MIN_BODY_CHARS
    except requests.RequestException as e:
        out["status"] = type(e).__name__  # record the error kind (Timeout, ConnectionError) alongside the HTTP codes
    return out


def run_pipeline(articles, max_workers=MAX_WORKERS):
    """Run process_article over every row in parallel, preserving order."""
    rows = articles.to_dict("records")
    results = [None] * len(rows)
    with make_session() as session, ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(process_article, row, session): i for i, row in enumerate(rows)}
        for f in as_completed(futs):
            results[futs[f]] = f.result()
    return pd.DataFrame(results)

## Running the pipeline

Run the composed pipeline over the open corpus, count what survives, and save the result.

### 6. Running across the open corpus

Drop the blocked sources and run over the open ones, around 2.7k articles.

In [19]:
open_articles = articles[articles["source"].isin(open_sources)].copy()

results = run_pipeline(open_articles)

merged = open_articles.merge(
    results[["article_id", "true_source", "utc", "raw_body", "processed_body",
             "status", "source_ok", "corrected", "time_source", "text_ok", "n_fetch"]],
    on="article_id", how="left",
)
print(f"{len(merged)} articles, {int(results['n_fetch'].sum())} fetches")

2742 articles, 2687 fetches


### 7. Results and losses

In [23]:
n = len(merged)
accept = merged["text_ok"]   # time always resolves (API floor), so text is the gate

print(f"open corpus          : {n}")
print(f"page reached (200)   : {merged['status'].eq(200).sum()}")
print(f"source recovered     : {merged['source_ok'].sum()}")
print(f"text recovered       : {merged['text_ok'].sum():>4}   reject {n - merged['text_ok'].sum()} on text")
print(f"ACCEPTED (text)      : {accept.sum():>4} ({accept.mean():.1%})")
print(f"LOST                 : {(~accept).sum():>4} ({(~accept).mean():.1%})")
print()

# On what grounds the lost articles were rejected: a reached page whose body was
# too short, versus a fetch that never returned a usable page (blocked, throttled, dead).
lost = merged[~accept]
reached_lost = lost["status"].eq(200)
print(f"lost, reached (200), body too short: {reached_lost.sum()}")
print(f"lost, never reached (error/non-200): {(~reached_lost).sum()}")
print("lost by status (HTTP code or error kind):")
print(lost["status"].value_counts(dropna=False).to_string())
print()

# Of the accepted rows, how many carry a scraped correction vs the API-time floor.
acc = merged[accept]
corr = int(acc["corrected"].sum())
print(f"time corrected (scraped): {corr:>4} ({corr / len(acc):.1%})")
print(f"time kept from API      : {len(acc) - corr:>4} ({1 - corr / len(acc):.1%})")

merged.assign(accepted=accept).groupby("source").agg(
    tried=("article_id", "size"), accepted=("accepted", "sum"))

open corpus          : 2742
page reached (200)   : 2146
source recovered     : 2146
text recovered       : 1818   reject 924 on text
ACCEPTED (text)      : 1818 (66.3%)
LOST                 :  924 (33.7%)

lost, reached (200), body too short: 328
lost, never reached (error/non-200): 596
lost by status (HTTP code or error kind):
status
200                328
RetryError         297
404                233
403                 56
401                  5
406                  4
ConnectionError      1

time corrected (scraped): 1751 (96.3%)
time kept from API      :   67 (3.7%)


,tried,accepted
source,,
Benzinga,451,450
DowJones,17,15
Yahoo,2274,1353


The status column carries either the HTTP code the server returned or, when the request never got a reply, the name of the error raised. What each of the losing values means:

- **RetryError**: not an HTTP code. The request was throttled (429) and used up its retry budget without ever getting through, so urllib3 gave up. This is the throttled bucket, and it is the only one worth chasing, since fewer workers or more bounded retries can recover it.
- **404 Not Found**: a dead link. The article was removed or the URL expired, common on old reposts.
- **403 Forbidden**: the server understood the request and refused it outright, from bot detection, a geo block, or a paywall. No login would change this.
- **401 Unauthorized**: the page needs a login we do not have. Gated content.
- **406 Not Acceptable**: the server rejected the request's headers, usually another shape of bot blocking.

The 401, 403, 404, and 406 rows are genuinely unfetchable and cap how much we can recover. Only RetryError is a tuning problem rather than a dead end.

### Chasing the throttled tail, one rerun at a time

The status breakdown splits the losses cleanly: the 404, 403, 401, and 406 rows are dead, but the throttled rows (RetryError) are only rate limited, articles Yahoo would serve if we came back when its per-IP window had refilled. Retrying them in the same run does not help, because the limiter clears with elapsed time, not with more requests, so a harder retry just waits inside the same closed window.

So instead of one long run we keep the results in memory and chase the tail across short reruns. The cell below looks at the current `results`, treats a row as settled once it has a body or a dead status, and refetches only the rows that are still pending. It folds the fresh attempts back into `results`, so the accepted count climbs each time and the rows that already landed are never fetched again.

The point is that no single run is long. Run the cell once now, and rerun it later when Yahoo is quieter, as many times as you like. Each pass only touches what is still missing, so it stays short and the corpus grows toward the ceiling the dead rows set.

In [24]:
# Rerun this cell to chase the throttled tail. A row is settled once it has a body
# or a dead status (401, 403, 404, 406); everything else is still pending. We
# refetch only the pending rows and fold them back into `results`, in memory, so
# the good rows are never fetched again and the accepted count rises each run.
DEAD = {401, 403, 404, 406}
settled = results["text_ok"] | results["status"].isin(DEAD)
pending_ids = results.loc[~settled, "article_id"]
print(f"settled: {int(settled.sum())}   pending: {len(pending_ids)}")

if len(pending_ids):
    retry_rows = open_articles[open_articles["article_id"].isin(pending_ids)].to_dict("records")
    with make_session() as session, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = [ex.submit(process_article, row, session) for row in retry_rows]
        refetched = pd.DataFrame([f.result() for f in as_completed(futs)])

    results = pd.concat(
        [results[~results["article_id"].isin(pending_ids)], refetched],
        ignore_index=True,
    )
    merged = open_articles.merge(
        results[["article_id", "true_source", "utc", "raw_body", "processed_body",
                 "status", "source_ok", "corrected", "time_source", "text_ok", "n_fetch"]],
        on="article_id", how="left",
    )
    gained = int(refetched["text_ok"].sum())
    print(f"recovered with body this run: {gained} of {len(pending_ids)}")
    print(f"accepted now                : {int(results['text_ok'].sum())} of {len(results)}")

settled: 2116   pending: 626
recovered with body this run: 2 of 626
accepted now                : 1820 of 2742


In [ ]:
# Losses after the second pass: the RetryError bucket should have shrunk, and the
# accepted count risen, while the dead 4xx rows stay put.
accept = merged["text_ok"]
lost = merged[~accept]

print(f"open corpus          : {len(merged)}")
print(f"ACCEPTED (text)      : {accept.sum():>4} ({accept.mean():.1%})")
print(f"LOST                 : {(~accept).sum():>4} ({(~accept).mean():.1%})")
print()
print("lost by status (HTTP code or error kind):")
print(lost["status"].value_counts(dropna=False).to_string())

The pipeline keeps 2,100 of the 2,742 open articles, 76.6%, and loses 642 (23.4%).

Text is the binding stage. 627 rows are rejected for no usable body, whether a dead link, a block, or a stub page, and only 15 more for a missing time on top of a good body. Reaching the page (89.1%), recovering the source (89.1%), and recovering the time (88.4%) all clear about 88 to 89 percent, so the body is what costs us.

Nearly all of the loss is Yahoo. Benzinga passes 449 of 451 and DowJones 15 of 17, while Yahoo passes 1,636 of 2,274, around 72%, its dead 404s and paywalled reposts carrying almost the entire loss.

The source label also opens up. The kept articles resolve to a wide set of real outlets, Benzinga, The Motley Fool, GuruFocus, 247wallst, Barchart, Zacks, Simply Wall St, Reuters, Investing.com and more, all previously flattened under the Yahoo label.

On speed, the full open corpus ran in 8.2 minutes at 0.18 seconds per article across 2,984 fetches, roughly one per article. The single fetch compose plus the 8 worker pool is what makes a corpus scale run practical.

### 8. The processed corpus

Kept articles become `processed_articles`: the raw schema with the true source, the corrected UTC time, and two added columns, `raw_body` (page HTML) and `processed_body` (clean text). Saved to `data/processed/processed_articles.parquet`.

In [ ]:
proc = merged[accept].copy()
proc["source"] = proc["true_source"].where(proc["source_ok"], proc["source"])
proc["timestamp_utc"] = proc["utc"]
processed_articles = proc[["article_id", "headline", "summary", "source", "url",
                           "timestamp_utc", "raw_body", "processed_body"]].reset_index(drop=True)

out = PROCESSED_DATA_DIR / "processed_articles.parquet"
processed_articles.to_parquet(out, compression="snappy")
print(f"saved {len(processed_articles)} rows to {out}")
processed_articles.drop(columns="raw_body").head()

### 9. Report

The final corpus in profile: which outlets it covers, when the articles were published, and how much text we recovered per article.

The kept corpus holds 2,100 articles drawn from 72 real outlets and spans roughly a year, from 22 August 2025 to 8 August 2026.

Benzinga leads with 659, many of them Yahoo reposts of Benzinga, then The Motley Fool at 296, GuruFocus at 152, and 247wallst at 118, trailing off into a long tail. The single Yahoo label from the raw corpus is now spread across dozens of named outlets.

Publishing clusters in the US market hours, peaking between 12 and 16 UTC, roughly the morning around the open. This is the signal the corrected UTC time protects.

The cleaned body runs a median of 3,445 characters, with a floor of 509 and a maximum of about 52k, far past the 50 token summary we started with.

In [ ]:
import matplotlib.pyplot as plt

report = pd.read_parquet(PROCESSED_DATA_DIR / "processed_articles.parquet")
report["hour"] = report["timestamp_utc"].dt.hour
report["day"] = report["timestamp_utc"].dt.date
report["body_len"] = report["processed_body"].str.len()

fig, ax = plt.subplots(2, 2, figsize=(13, 9))

top = report["source"].value_counts().head(12)
ax[0, 0].barh(top.index[::-1], top.values[::-1], color="#4C72B0")
ax[0, 0].set_title(f"Top outlets among {len(report)} kept articles")
ax[0, 0].set_xlabel("articles")

daily = report.groupby("day").size()
ax[0, 1].plot(list(daily.index), daily.values, color="#55A868")
ax[0, 1].set_title("Articles per day")
ax[0, 1].tick_params(axis="x", rotation=45)

hours = report["hour"].value_counts().reindex(range(24), fill_value=0)
ax[1, 0].bar(range(24), hours.values, color="#C44E52")
ax[1, 0].set_title("Publish hour (UTC)")
ax[1, 0].set_xlabel("hour of day")

ax[1, 1].hist(report["body_len"].clip(upper=15000), bins=40, color="#8172B3")
ax[1, 1].set_title("Cleaned body length (chars, clipped at 15k)")
ax[1, 1].set_xlabel("characters")

plt.tight_layout()
plt.show()

This closes the text-recovery notebook. We began with a raw corpus of short summaries, a misleading source label, and untrustworthy times, and we end with a filtered corpus carrying the true outlet, a corrected UTC time, and a full cleaned body per article. The next notebook takes `processed_articles` into the entity filter and FinBERT sentiment scoring.